# 02 — Judge Scoring

Run all 50 evaluation instances through each of the three Ollama judge models and record
the raw scores. Wall-clock time is reported per judge to confirm the ~65-minute total estimate.

**Prerequisites:**
- Notebook 01 has been run (answer corpus and human scores exist in `data/`)
- Ollama is running: `ollama serve` (handled by `make setup`)
- Models are pulled: `make setup` pulls `qwen2.5:1.5b`, `qwen2.5:3b`, `gemma3:4b`

**Outputs produced:**
- `data/eval/scores_qwen2_5_1_5b.json`
- `data/eval/scores_qwen2_5_3b.json`
- `data/eval/scores_gemma3_4b.json`

> **RAM constraint:** Models are scored sequentially. `gemma3:4b` requires ~3.8 GB — do not
> run two judges simultaneously in the same Codespace.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import json
import time

import pandas as pd

from src.config import load_settings
from src.judging.judge import OllamaJudge
from src.judging.runner import METRICS

settings = load_settings(ROOT / "config" / "settings.yaml")
print(f"Ollama URL: {settings.ollama_url}")
print(f"Models: {settings.models}")

FIGURES_DIR = ROOT / "outputs" / "figures"
RESULTS_DIR = ROOT / "outputs" / "results"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

## 1. Load answer corpus

In [ ]:
answers_dir = ROOT / "data" / "answers"

all_instances = []
for path in sorted(answers_dir.glob("*.json")):
    data = json.loads(path.read_text())
    if isinstance(data, list):
        all_instances.extend(data)
    else:
        all_instances.append(data)

print(f"Loaded {len(all_instances)} instances from {answers_dir}")
pd.DataFrame(all_instances).groupby(["pipeline", "question_type"]).size().rename("count").reset_index()

## 2. Check Ollama availability

In [ ]:
import httpx

try:
    resp = httpx.get(f"{settings.ollama_url}/api/tags", timeout=5.0)
    available_models = [m["name"] for m in resp.json().get("models", [])]
    print(f"Ollama is running. Available models: {available_models}")
except Exception as e:
    print(f"Ollama not reachable: {e}")
    print("Run: ollama serve  (or: make setup)")
    available_models = []

## 3. Score all instances

Each model is scored independently. Results are appended to a list and flushed to disk
after each model so progress is not lost on interruption.

Estimated times on a 2-CPU Codespace:
- `qwen2.5:1.5b`: ~15 min
- `qwen2.5:3b`: ~20 min  
- `gemma3:4b`: ~30 min

In [ ]:
def score_model(model: str, instances: list, output_dir: Path) -> list:
    """Score all instances with one judge. Returns list of score records."""
    judge = OllamaJudge(model=model, ollama_url=settings.ollama_url)
    records = []
    total = len(instances) * len(METRICS)
    done = 0
    t0 = time.perf_counter()

    for inst_idx, instance in enumerate(instances, 1):
        for metric in METRICS:
            try:
                score = judge.score(
                    metric=metric,
                    question=str(instance["question"]),
                    context=instance["context"],
                    answer=str(instance["answer"]),
                )
            except Exception as exc:
                print(f"  ERROR {instance['id']} / {metric}: {exc}")
                score = float("nan")
            records.append({"id": instance["id"], "model": model, "metric": metric, "score": score})
            done += 1
            elapsed = time.perf_counter() - t0
            rate = done / elapsed if elapsed > 0 else 0
            eta = (total - done) / rate if rate > 0 else 0
            print(f"  [{model}] instance {inst_idx}/{len(instances)}, {metric}: score={score:.3f}  "
                  f"({done}/{total} total, {elapsed:.0f}s elapsed, ETA {eta:.0f}s)")

    elapsed = time.perf_counter() - t0
    safe = model.replace(":", "_").replace(".", "_")
    out_path = output_dir / f"scores_{safe}.json"
    out_path.write_text(json.dumps(records, indent=2))
    print(f"  Saved {len(records)} records to {out_path}  ({elapsed:.1f}s total)")
    return records

In [ ]:
eval_dir = ROOT / "data" / "eval"
eval_dir.mkdir(parents=True, exist_ok=True)

timing = {}

for model in settings.models:
    safe = model.replace(":", "_").replace(".", "_")
    out_path = eval_dir / f"scores_{safe}.json"

    if out_path.exists():
        existing = json.loads(out_path.read_text())
        if existing:
            print(f"SKIP {model}: {len(existing)} records already in {out_path.name}")
            continue
        out_path.unlink()  # remove empty file and re-score

    if model not in available_models:
        print(f"SKIP {model}: not available in Ollama (pull with: ollama pull {model})")
        continue

    print(f"\nScoring with {model} ({len(all_instances)} instances × {len(METRICS)} metrics)...")
    t0 = time.perf_counter()
    score_model(model, all_instances, eval_dir)
    timing[model] = time.perf_counter() - t0

if timing:
    print("\n=== Timing summary ===")
    for m, t in timing.items():
        print(f"  {m}: {t/60:.1f} min")
else:
    print("\nAll models skipped (already scored or unavailable).")

## 4. Score summary

In [ ]:
from IPython.display import display

score_files = list(eval_dir.glob("scores_*.json"))
if not score_files:
    print("No score files found. Run the scoring cell above first.")
else:
    all_records = []
    for f in sorted(score_files):
        all_records.extend(json.loads(f.read_text()))

    if not all_records:
        print("Score files found but all are empty — run the scoring cell above.")
    else:
        scores_df = pd.DataFrame(all_records)
        print(f"Total score records: {len(scores_df)}")
        print(f"Models scored: {sorted(scores_df['model'].unique())}")
        print("\nMean score per (model, metric):")
        display(scores_df.groupby(["model", "metric"])["score"].mean().unstack().round(3))

        summary = scores_df.groupby(["model", "metric"])["score"].agg(["mean", "std", "count"]).round(4)
        summary_path = RESULTS_DIR / "02_score_summary.json"
        summary.reset_index().to_json(summary_path, orient="records", indent=2)
        print(f"Saved score summary to {summary_path}")

## 5. Load and inspect human scores

In [ ]:
human_path = ROOT / "data" / "human" / "human_scores.csv"
if human_path.exists():
    human_df = pd.read_csv(human_path)
    print(f"Human scores: {len(human_df)} instances")
    print("\nMean human scores by metric:")
    print(human_df[["context_relevance", "groundedness", "answer_relevance"]].mean().round(3).to_string())
else:
    print("Human scores not found — run notebook 01 first.")

---
**Next:** Run `03_inter_judge_agreement.ipynb` to compute kappa and correlation metrics across all annotators.